In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.ndimage import gaussian_filter
import pandas as pd
import scipy.optimize
import math
import MDAnalysis as md

def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return (average, math.sqrt(variance))


path = '/Volumes/Elements/PTM_project/FLNC24/METAD/'

In [28]:
u = md.Universe(path+'processed.pdb',path+'fit1.xtc')

In [33]:
# find frames where SASA closest to 0.65 and cis
sim=1
SASA = np.loadtxt(path+'RMSD_SASA_data/sasaSER2718_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]

t= 1000
t_discard = 200


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)<0.52)[0] # 30 deg cutoff

targetSASA = 0.65
diff = np.abs(SASA-targetSASA)
mindiff = np.min(diff)
minidx = np.where(diff==mindiff)[0]

frames = np.intersect1d(minidx,cisframes)

print(time[frames[5]])
print(zeta[frames[5]])
print(SASA[frames[5]])

298065.014157
0.180139
0.65


In [34]:
frames

array([ 28683,  29652,  31584,  31610,  31717,  59613,  94505, 101752,
       102007, 102021, 102442, 104098, 108369, 116687, 121835, 122453,
       186339, 186425, 186829, 187185, 187314])

In [35]:
print(frames[5])

59613


In [36]:
frame=59613
u.trajectory[frame]
protein = u.select_atoms('all')
with md.Writer(path+'flnc24_sim1_{}ps_highSASA_cis.pdb'.format(int(time[frames[5]])),protein.n_atoms) as W:
    W.write(protein)
print('Done')

Done


In [37]:
# find frames where SASA closest to 0.8 and trans
sim=1
SASA = np.loadtxt(path+'RMSD_SASA_data/sasaSER2718_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]

t= 1000
t_discard = 200


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)>2.62)[0] # 150 deg cutoff

targetSASA = 0.8
diff = np.abs(SASA-targetSASA)
mindiff = np.min(diff)
minidx = np.where(diff==mindiff)[0]

frames = np.intersect1d(minidx,cisframes)

print(time[frames[5]])
print(zeta[frames[5]])
print(SASA[frames[5]])

419645.019932
-2.94431
0.8


In [38]:
frames

array([ 37593,  39692,  43653,  45336,  81913,  83929,  84242,  85639,
       191708, 192915, 196097, 196299, 196414, 197594])

In [39]:
frame=83929
u.trajectory[frame]
protein = u.select_atoms('all')
with md.Writer(path+'flnc24_sim1_{}ps_highSASA_trans.pdb'.format(int(time[frames[5]])),protein.n_atoms) as W:
    W.write(protein)
print('Done')

Done
